In [1]:
# Cell 1 — Setup + data load
# Binance hourly (2021-04-01+) + Deribit BTC DVOL + PIT universe + entry signals + DataPanel

import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests

for _eng in [
    Path(r"C:\Users\User\backtest_engine"),
    Path("/content/drive/MyDrive/backtest_engine"),
    Path("/content/backtest_engine"),
    Path("..").resolve() / "backtest_engine",
]:
    if (_eng / "backtest").is_dir():
        sys.path.insert(0, str(_eng))
        break
else:
    raise FileNotFoundError("backtest_engine not found")

_nb_dir = str(Path.cwd())
if _nb_dir not in sys.path:
    sys.path.insert(0, _nb_dir)

from backtest.data import DataPanel, FieldSpec

# ── Parameters ────────────────────────────────────────────────────────────────
FAST_MODE        = True
CRYPTO_TOP_N     = 20 if FAST_MODE else 50
W_LIQ            = 168   # trailing qvol lookback for PIT universe ranking (hours)
T_TREND          = 168   # uptrend cumulative log-return window (hours)
R_MIN            = 0.06  # minimum cumulative log return for trend_up
L_MA             = 24    # SMA window for pullback signal (hours)
RECENT_DOWN_BARS = 1     # require negative log return on last N bar(s)
MIN_HISTORY_FRAC = 0.9

START_UTC = pd.Timestamp("2021-04-01", tz="UTC")   # first date Deribit DVOL is available
END_UTC   = pd.Timestamp.now(tz="UTC").floor("h")

STABLE_EXCLUDE = frozenset({
    "USDTUSDT", "USDCUSDT", "USDSUSDT", "USDEUSDT", "USD1USDT", "BFUSDUSDT",
    "UUSDT", "RLUSDUSDT", "PAXGUSDT", "XAUTUSDT", "EURUSDT", "FDUSDUSDT",
    "TUSDUSDT", "USDPUSDT",
})

CACHE_DIR     = Path("data")
BINANCE_CACHE = CACHE_DIR / "binance_hourly_top50_pool.parquet"
DVOL_CACHE    = CACHE_DIR / "btc_dvol_hourly.parquet"
MATIC_TO_POL  = pd.Timestamp("2024-09-04", tz="UTC")

ANCHOR_BINANCE = [
    "BTCUSDT", "ETHUSDT", "BNBUSDT", "SOLUSDT", "XRPUSDT",
    "DOGEUSDT", "ADAUSDT", "SHIBUSDT", "AVAXUSDT", "DOTUSDT",
    "LINKUSDT", "TRXUSDT", "BCHUSDT", "NEARUSDT", "MATICUSDT",
    "UNIUSDT", "LTCUSDT", "ICPUSDT", "ETCUSDT", "HBARUSDT",
    "ATOMUSDT", "FILUSDT", "APTUSDT", "ARBUSDT", "OPUSDT",
    "INJUSDT", "SUIUSDT", "SEIUSDT", "TIAUSDT", "LDOUSDT",
]

# ── Deribit DVOL fetch ────────────────────────────────────────────────────────
_DVOL_URL = "https://www.deribit.com/api/v2/public/get_volatility_index_data"

def _fetch_dvol(start_dt, end_dt, currency="BTC", resolution="3600", pause=0.15):
    start_ms, end_ms = int(start_dt.timestamp() * 1000), int(end_dt.timestamp() * 1000)
    candles, cur_end = [], end_ms
    while True:
        r = requests.get(
            _DVOL_URL,
            params={"currency": currency, "start_timestamp": start_ms,
                    "end_timestamp": cur_end, "resolution": resolution},
            timeout=15,
        )
        r.raise_for_status()
        res = r.json()["result"]
        candles.extend(res["data"])
        if res["continuation"] is None:
            break
        cur_end = res["continuation"]
        time.sleep(pause)
    candles.sort(key=lambda x: x[0])
    df = pd.DataFrame(candles, columns=["ts_ms", "open", "high", "low", "close"])
    df.index = pd.to_datetime(df.pop("ts_ms"), unit="ms", utc=True)
    return df

if DVOL_CACHE.exists():
    dvol_raw = pd.read_parquet(DVOL_CACHE)
    _last = dvol_raw.index.max()
    _last_utc = _last if _last.tzinfo else _last.tz_localize("UTC")
    if END_UTC - _last_utc > pd.Timedelta(hours=2):
        _new = _fetch_dvol(_last_utc - pd.Timedelta(hours=1), END_UTC)
        dvol_raw = pd.concat([dvol_raw, _new]).sort_index()
        dvol_raw = dvol_raw[~dvol_raw.index.duplicated(keep="last")]
        dvol_raw.to_parquet(DVOL_CACHE)
        print(f"DVOL cache updated → {dvol_raw.index.max()}")
    else:
        print(f"DVOL cache up-to-date through {_last}")
else:
    print("Fetching Deribit BTC DVOL (2021-04-01 → now) …")
    dvol_raw = _fetch_dvol(START_UTC, END_UTC)
    dvol_raw.to_parquet(DVOL_CACHE)
    print(f"DVOL cached: {len(dvol_raw):,} bars  {dvol_raw.index.min()} → {dvol_raw.index.max()}")

# ── Load Binance hourly parquet ───────────────────────────────────────────────
if not BINANCE_CACHE.exists():
    raise FileNotFoundError(f"Missing {BINANCE_CACHE}. Run ou_optimal_trend_pullback.ipynb Cell 0 first.")

raw  = pd.read_parquet(BINANCE_CACHE)
raw["open_time"] = pd.to_datetime(raw["open_time"], utc=True)

matic = raw[raw["symbol"] == "MATICUSDT"]
pol   = raw[raw["symbol"] == "POLUSDT"]
if not matic.empty and not pol.empty:
    stitched = pd.concat(
        [matic[matic["open_time"] < MATIC_TO_POL],
         pol[pol["open_time"] >= MATIC_TO_POL]],
        ignore_index=True,
    ).assign(symbol="MATICUSDT")
    raw = pd.concat([raw[~raw["symbol"].isin(["MATICUSDT", "POLUSDT"])], stitched], ignore_index=True)

trade_cols = sorted({s for s in raw["symbol"].unique() if s != "POLUSDT"})
close = raw.pivot(index="open_time", columns="symbol", values="close").reindex(columns=trade_cols)
qvol  = raw.pivot(index="open_time", columns="symbol", values="quote_volume").reindex(columns=trade_cols)

full_idx = pd.date_range(close.index.min(), close.index.max(), freq="1h", tz="UTC")
close = close.reindex(full_idx)
qvol  = qvol.reindex(full_idx)
# strip timezone — DataPanel requires a plain DatetimeIndex (no tz)
close.index = close.index.tz_convert("UTC").tz_localize(None)
qvol.index  = qvol.index.tz_convert("UTC").tz_localize(None)

# ── Filter to DVOL start date ─────────────────────────────────────────────────
_start_naive = START_UTC.tz_localize(None)
close = close[close.index >= _start_naive].copy()
qvol  = qvol[qvol.index  >= _start_naive].copy()

# ── Process DVOL → tz-naive series aligned to close.index ────────────────────
_dvol_s = dvol_raw["close"].copy()
if _dvol_s.index.tzinfo is not None:
    _dvol_s.index = _dvol_s.index.tz_convert("UTC").tz_localize(None)
dvol_aligned = _dvol_s.reindex(close.index, method="ffill").rename("DVOL")

# ── Monthly PIT universe (top-N by trailing qvol, decided at prior month-end) ─
MIN_HISTORY_BARS = max(int(W_LIQ * MIN_HISTORY_FRAC), 24)
_rank_cols   = [c for c in close.columns if c not in STABLE_EXCLUDE]
pit_universe = pd.DataFrame(False, index=close.index, columns=close.columns)

for _per in close.index.to_period("M").unique():
    _mask = close.index.to_period("M") == _per
    _idx  = np.flatnonzero(_mask)
    if _idx[0] == 0:
        continue
    _t_dec  = close.index[_idx[0] - 1]
    _px_lb  = close.loc[:_t_dec, _rank_cols].iloc[-W_LIQ:]
    _vol_lb = qvol.loc[:_t_dec, _rank_cols].iloc[-W_LIQ:]
    _n_ok   = _px_lb.notna().sum()
    _last   = _px_lb.iloc[-1]
    _elig   = (_n_ok >= MIN_HISTORY_BARS) & _last.notna() & _vol_lb.notna().any()
    _liq    = _vol_lb.median()
    _liq[~_elig] = np.nan
    _ranks  = _liq.rank(ascending=False, method="min")
    _picked = _ranks.index[_ranks <= CRYPTO_TOP_N]
    pit_universe.loc[_mask, _picked] = True

# ── Causal entry features ─────────────────────────────────────────────────────
_min_trend = max(T_TREND // 2, 24)
_min_ma    = max(L_MA // 2, 12)

log_ret    = np.log(close / close.shift(1))
roll_trend = log_ret.rolling(T_TREND, min_periods=_min_trend).sum()
trend_up   = roll_trend > R_MIN
sma        = close.rolling(L_MA, min_periods=_min_ma).mean()
pullback   = close < sma
recent_down = log_ret < 0 if RECENT_DOWN_BARS == 1 else \
              log_ret.rolling(RECENT_DOWN_BARS, min_periods=RECENT_DOWN_BARS).sum() < 0
entry_signal = pit_universe & trend_up & pullback & recent_down

# ── Broadcast DVOL scalar to all asset columns ────────────────────────────────
dvol_broadcast = pd.DataFrame(
    np.repeat(dvol_aligned.values[:, None], len(close.columns), axis=1),
    index=close.index,
    columns=close.columns,
    dtype=np.float64,
)

# ── DataPanel ─────────────────────────────────────────────────────────────────
features = {
    "pit_universe": pit_universe.astype(np.float64),
    "trend_up":     trend_up.astype(np.float64),
    "pullback":     pullback.astype(np.float64),
    "recent_down":  recent_down.astype(np.float64),
    "entry_signal": entry_signal.astype(np.float64),
    "roll_trend":   roll_trend.astype(np.float64),
    "sma_pullback": sma.astype(np.float64),
    "log_returns":  log_ret.astype(np.float64),
    "DVOL":         dvol_broadcast,
}

_feat_spec = FieldSpec(lag=0, missing="drop")
specs = {
    "prices":   FieldSpec(lag=0),
    "volume":   FieldSpec(lag=0),
    "universe": FieldSpec(lag=0),
    **{k: _feat_spec for k in features},
}

panel = DataPanel(
    prices=close,
    volume=qvol,
    features=features,
    universe=pit_universe.astype(bool),
    specs=specs,
    check_outliers=True,
)

# ── Validation output ─────────────────────────────────────────────────────────
_n_entry = int(entry_signal.sum().sum())
_n_pit   = int(pit_universe.sum().sum())
print(f"-- DataPanel (crypto_regime_ou_trend) --")
print(f"FAST_MODE          : {FAST_MODE}  (CRYPTO_TOP_N={CRYPTO_TOP_N})")
print(f"Bars               : {len(panel.dates):,}")
print(f"Assets in matrix   : {len(panel.assets_all)}")
print(f"Range              : {panel.dates[0]} → {panel.dates[-1]}")
print(f"NaN fraction prices: {close.isna().mean().mean():.2%}")
print(f"DVOL bars          : {dvol_aligned.notna().sum():,}  (NaN after ffill: {dvol_aligned.isna().sum():,})")
print(f"DVOL range         : {dvol_aligned.min():.1f} → {dvol_aligned.max():.1f}")
print(f"PIT universe cells : {_n_pit:,}")
print(f"Entry signal cells : {_n_entry:,}")
print(f"Fields             : {panel.available_fields()}")
print(f"\nEntry signals per year (sum over assets):")
print(entry_signal.sum(axis=1).resample("YE").sum().to_string())
print()
close[[c for c in ["BTCUSDT", "ETHUSDT", "SOLUSDT"] if c in close.columns]].tail(3)


Fetching Deribit BTC DVOL (2021-04-01 → now) …


DVOL cached: 45,167 bars  2021-04-01 00:00:00+00:00 → 2026-05-26 22:00:00+00:00


C:\Users\User\backtest_engine\backtest\data\panel.py:172: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  rets = self._prices.pct_change()


-- DataPanel (crypto_regime_ou_trend) --
FAST_MODE          : True  (CRYPTO_TOP_N=20)
Bars               : 45,035
Assets in matrix   : 65
Range              : 2021-04-01 00:00:00 → 2026-05-21 10:00:00
NaN fraction prices: 32.88%
DVOL bars          : 45,035  (NaN after ffill: 0)
DVOL range         : 31.5 → 166.4
PIT universe cells : 886,300
Entry signal cells : 49,286
Fields             : ['prices', 'volume', 'DVOL', 'entry_signal', 'log_returns', 'pit_universe', 'pullback', 'recent_down', 'roll_trend', 'sma_pullback', 'trend_up']

Entry signals per year (sum over assets):
2021-12-31     8091
2022-12-31     8423
2023-12-31     9881
2024-12-31    11660
2025-12-31     8857
2026-12-31     2374
Freq: YE-DEC



C:\Users\User\AppData\Local\Temp\ipykernel_42372\1321591115.py:209: UserWarning: DataPanel: 13822 bar(s) with |return - median| > 10.0*MAD detected.
  Top examples:
    2025-10-10 USDEUSDT: ret=-6.35%, mad_score=635.5
    2025-10-10 USDEUSDT: ret=+6.05%, mad_score=605.9
    2024-04-13 PAXGUSDT: ret=+19.83%, mad_score=218.0
    2024-04-13 PAXGUSDT: ret=-13.76%, mad_score=151.2
    2021-07-24 DEXEUSDT: ret=+69.51%, mad_score=149.2
  Verify these are real moves, not bad ticks. Pass check_outliers=False to silence.
  panel = DataPanel(


symbol,BTCUSDT,ETHUSDT,SOLUSDT
2026-05-21 08:00:00,NaN,NaN,NaN
2026-05-21 09:00:00,NaN,NaN,NaN
2026-05-21 10:00:00,NaN,NaN,NaN


In [ ]:
# Cell 2 — DVOL calibration reference + strategy parameters
# DVOL_PERCENTILE / DVOL_WARMUP_BARS feed directly into the strategy (Cell 3).
# The expanding percentile is computed live inside generate_weights — fully PIT-safe.

import numpy as np

# ── Strategy parameters ───────────────────────────────────────────────────────
DVOL_PERCENTILE  = 65    # regime OFF when current DVOL > P65 of all DVOL seen so far
DVOL_WARMUP_BARS = 720   # 30 days × 24 h — gate inactive until this many bars seen

# ── Reference: full-sample distribution (context only, NOT used by the strategy) ─
_vals = dvol_aligned.dropna().values
print("BTC DVOL full-sample distribution (reference only):")
for _p in [10, 25, 50, 65, 75, 85, 90, 95, 99]:
    print(f"  p{_p:2d}: {np.percentile(_vals, _p):.1f}")
print(f"\n  mean : {_vals.mean():.1f}   std : {_vals.std():.1f}")
print(f"  min  : {_vals.min():.1f}   max : {_vals.max():.1f}")

_p65_full = np.percentile(_vals, DVOL_PERCENTILE)
print(f"\nFull-sample p{DVOL_PERCENTILE}: {_p65_full:.1f}  "
      f"(~{(_vals > _p65_full).mean():.0%} of bars would be regime-OFF under this fixed level)")

# ── Expanding-percentile trace — shows how the threshold evolves PIT-style ────
import pandas as pd
_dvol_daily = dvol_aligned.resample("D").last().dropna()
_exp_thresh  = [
    float(np.percentile(_dvol_daily.values[:i+1], DVOL_PERCENTILE))
    for i in range(len(_dvol_daily))
]
_trace = pd.Series(_exp_thresh, index=_dvol_daily.index, name="expanding_p65")

print(f"\nExpanding p{DVOL_PERCENTILE} — first/last 5 daily values:")
print(_trace.head())
print(_trace.tail())
print(f"\nGate warmup: {DVOL_WARMUP_BARS} bars  "
      f"(≈ {DVOL_WARMUP_BARS/24:.0f} days — regime ON until then regardless of DVOL)")
print(f"\nDVOL_PERCENTILE  = {DVOL_PERCENTILE}")
print(f"DVOL_WARMUP_BARS = {DVOL_WARMUP_BARS}")


In [ ]:
# Cell 3 â€” CryptoRegimeOuTrendStrategy
# Layer 1: BTC DVOL expanding-percentile macro gate (regime OFF â†’ close all, block entries)
# Layer 2: OuTrendPullbackStrategy (trend+pullback entry, ABM/GBM OU-optimal exit, per-asset CPD)

import os, subprocess, sys, time
from pathlib import Path

try:
    from numba import njit, prange
    HAS_NUMBA = True
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "numba"])
    from numba import njit, prange
    HAS_NUMBA = True

N_CORES = os.cpu_count() or 8
os.environ["NUMBA_NUM_THREADS"] = str(N_CORES)
os.environ.setdefault("OMP_NUM_THREADS", str(N_CORES))

import numpy as np
import pandas as pd
from backtest.strategy import Strategy
from backtest.risk.config import RiskConfig
from _mood_helpers_ou_optimal_trend_pullback import _mood_d_max_tau, MOOD_THRESHOLD

_FAST  = FAST_MODE
_NUM_PATHS  = 10_000 if _FAST else 100_000
_MAX_HOLD   = 72
_OLS_WINDOW = 168
_MAX_POSITIONS           = 10
_WEIGHT_PER_POSITION     = 0.05
_MAX_NEW_ENTRIES_PER_BAR = 3
_MIN_PI_WIDTH        = 5.0
_SIGMA_PERCENTILE    = 0.75
_PI_PLUS_GRID  = np.arange(0.5,  10.0 + 0.5 * 0.5, 0.5, dtype=np.float64)
_PI_MINUS_GRID = np.arange(-10.0, -0.5 + 0.5 * 0.5, 0.5, dtype=np.float64)

# â”€â”€ Numba JIT functions â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

@njit(cache=True)
def _fit_ou_numba(s, m):
    n = len(s)
    if n < 10: return np.nan, np.nan
    y = np.empty(n - 1); x = np.empty(n - 1)
    for i in range(n - 1): y[i] = s[i+1] - s[i]; x[i] = s[i] - m
    mx = 0.0; my = 0.0
    for i in range(n - 1): mx += x[i]; my += y[i]
    mx /= n-1; my /= n-1
    var_x = 0.0; cov_xy = 0.0
    for i in range(n - 1):
        dx = x[i]-mx; dy = y[i]-my
        var_x += dx*dx; cov_xy += dx*dy
    if n-1 > 1: var_x /= n-2; cov_xy /= n-2
    if var_x < 1e-14: return np.nan, np.nan
    phi = 1.0 + cov_xy / var_x
    ss = 0.0
    for i in range(n - 1):
        resid = y[i] - (phi - 1.0)*x[i]; ss += resid*resid
    sigma = np.sqrt(ss/(n-2)) if n > 2 else np.nan
    if not np.isfinite(phi) or not np.isfinite(sigma) or sigma <= 0.0: return np.nan, np.nan
    if phi <= -1.0 or phi >= 1.0: return np.nan, np.nan
    return phi, sigma

@njit(cache=True)
def _detect_trend_type(s):
    n = len(s)
    if n < 10: return 'abm'
    x = np.arange(n, dtype=np.float64); mx = np.mean(x); ms = np.mean(s)
    cov = 0.0; var_x = 0.0
    for i in range(n):
        dx = x[i]-mx; ds = s[i]-ms; cov += dx*ds; var_x += dx*dx
    slope_linear = cov/var_x if var_x > 0 else 0.0
    ss_res = 0.0; ss_tot = 0.0
    for i in range(n):
        pred = ms + slope_linear*(x[i]-mx)
        ss_res += (s[i]-pred)**2; ss_tot += (s[i]-ms)**2
    r2_linear = 1.0 - ss_res/ss_tot if ss_tot > 0 else 0.0
    log_s = np.log(s); m_log = np.mean(log_s)
    cov_log = 0.0
    for i in range(n): cov_log += (x[i]-mx)*(log_s[i]-m_log)
    slope_log = cov_log/var_x if var_x > 0 else 0.0
    ss_res_log = 0.0; ss_tot_log = 0.0
    for i in range(n):
        pred_log = m_log + slope_log*(x[i]-mx)
        ss_res_log += (log_s[i]-pred_log)**2; ss_tot_log += (log_s[i]-m_log)**2
    r2_log = 1.0 - ss_res_log/ss_tot_log if ss_tot_log > 0 else 0.0
    return 'gbm' if r2_log > r2_linear else 'abm'

@njit(cache=True)
def _fit_abm_numba(s):
    n = len(s)
    if n < 10: return np.nan, np.nan
    deltas = np.empty(n-1)
    for i in range(n-1): deltas[i] = s[i+1] - s[i]
    mu = 0.0
    for i in range(n-1): mu += deltas[i]
    mu /= n-1
    var = 0.0
    for i in range(n-1): diff = deltas[i]-mu; var += diff*diff
    sigma = np.sqrt(var/(n-2)) if n > 2 else np.nan
    if not np.isfinite(mu) or not np.isfinite(sigma) or sigma <= 0.0: return np.nan, np.nan
    return mu, sigma

@njit(cache=True)
def _fit_gbm_numba(s):
    n = len(s)
    if n < 10: return np.nan, np.nan
    log_returns = np.empty(n-1)
    for i in range(n-1):
        if s[i] <= 0 or s[i+1] <= 0: return np.nan, np.nan
        log_returns[i] = np.log(s[i+1]/s[i])
    mu = 0.0
    for i in range(n-1): mu += log_returns[i]
    mu /= n-1
    var = 0.0
    for i in range(n-1): diff = log_returns[i]-mu; var += diff*diff
    sigma = np.sqrt(var/(n-2)) if n > 2 else np.nan
    if not np.isfinite(mu) or not np.isfinite(sigma) or sigma <= 0.0: return np.nan, np.nan
    return mu, sigma

@njit(parallel=True, cache=True)
def _calibrate_trend_fused(mu, sigma, s0, n_paths, max_h, seed, pi_plus_grid, pi_minus_grid, model_type):
    np.random.seed(seed & 0xFFFFFFFF)
    n_steps = max_h + 1
    pi_paths = np.empty((n_paths, n_steps))
    for i in prange(n_paths):
        si = s0; pi_paths[i, 0] = 0.0
        for step in range(1, n_steps):
            eps = np.random.randn()
            if model_type == 0: si = si + mu + sigma*eps
            else:               si = si * np.exp((mu - 0.5*sigma*sigma) + sigma*eps)
            pi_paths[i, step] = si - s0
    n_plus = pi_plus_grid.shape[0]; n_minus = pi_minus_grid.shape[0]
    best_sharpe = -1.0e300; best_plus = pi_plus_grid[0]; best_minus = pi_minus_grid[0]
    for ip in range(n_plus):
        pi_p = pi_plus_grid[ip]
        for im in range(n_minus):
            pi_m = pi_minus_grid[im]
            sum_p = 0.0; sum_sq = 0.0
            for i in range(n_paths):
                exit_idx = max_h
                for j in range(1, n_steps):
                    pi = pi_paths[i, j]
                    if pi >= pi_p or pi <= pi_m: exit_idx = j; break
                pnl = pi_paths[i, exit_idx]; sum_p += pnl; sum_sq += pnl*pnl
            mean = sum_p/n_paths
            var = (sum_sq - n_paths*mean*mean)/(n_paths-1) if n_paths > 1 else 0.0
            if var < 1e-24: continue
            sharpe = mean/np.sqrt(var)
            if sharpe > best_sharpe: best_sharpe = sharpe; best_plus = pi_p; best_minus = pi_m
    if best_sharpe <= -1.0e299: return np.nan, np.nan, np.nan
    return best_plus, best_minus, best_sharpe

# â”€â”€ GPU fallback â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
_USE_GPU = os.environ.get("USE_GPU", "auto").lower()
_cp = None; HAS_GPU = False

def _try_init_gpu():
    global _cp, HAS_GPU
    if _USE_GPU == "false": return
    try:
        import cupy as cp
    except ImportError:
        if _USE_GPU == "true":
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "cupy-cuda12x"])
            import cupy as cp
        else: return
    if cp.cuda.is_available(): _cp = cp; HAS_GPU = True

_try_init_gpu()

def _calibrate_trend_gpu(mu, sigma, s0, n_paths, max_h, seed, pi_plus_grid, pi_minus_grid, model_type):
    cp = _cp; rng = cp.random.RandomState(seed & 0xFFFFFFFF)
    n_paths = int(n_paths); max_h = int(max_h); n_steps = max_h + 1
    eps = rng.standard_normal((n_paths, max_h), dtype=cp.float64)
    pi_paths = cp.empty((n_paths, n_steps), dtype=cp.float64); pi_paths[:, 0] = 0.0
    if model_type == 0:
        cum_s = s0 + cp.cumsum(mu + sigma*eps, axis=1); pi_paths[:, 1:] = cum_s - s0
    else:
        log_inc = (mu - 0.5*sigma*sigma) + sigma*eps
        cum_s = s0 * cp.exp(cp.cumsum(log_inc, axis=1)); pi_paths[:, 1:] = cum_s - s0
    pi = pi_paths[:, 1:]; best_sharpe = -1.0e300
    best_plus = float(pi_plus_grid[0]); best_minus = float(pi_minus_grid[0])
    for pi_p in pi_plus_grid:
        for pi_m in pi_minus_grid:
            hit = (pi >= pi_p) | (pi <= pi_m); has_hit = hit.any(axis=1)
            first_col = hit.argmax(axis=1)
            exit_col = cp.where(has_hit, first_col+1, max_h)
            pnl = pi_paths[cp.arange(n_paths), exit_col]
            mean = pnl.mean(); var = pnl.var(ddof=1) if n_paths > 1 else cp.float64(0.0)
            if float(var) < 1e-24: continue
            sharpe = float(mean/cp.sqrt(var))
            if sharpe > best_sharpe: best_sharpe = sharpe; best_plus = float(pi_p); best_minus = float(pi_m)
    if best_sharpe <= -1.0e299: return np.nan, np.nan, np.nan
    return best_plus, best_minus, best_sharpe

# â”€â”€ OuTrendPullbackStrategy (base) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

class OuTrendPullbackStrategy(Strategy):
    """Trend + pullback entry + OU-optimal exit thresholds + per-asset CUSUM CPD."""
    rebalance_frequency = "D"

    def __init__(self, fast_mode=_FAST, num_simulated_paths=None, max_holding_period=_MAX_HOLD,
                 ou_ols_window=_OLS_WINDOW, max_positions=_MAX_POSITIONS,
                 weight_per_position=_WEIGHT_PER_POSITION, max_new_entries_per_bar=_MAX_NEW_ENTRIES_PER_BAR,
                 pi_plus_grid=None, pi_minus_grid=None, min_pi_width=_MIN_PI_WIDTH,
                 sigma_percentile=_SIGMA_PERCENTILE, mood_threshold=MOOD_THRESHOLD,
                 mood_warmup=21, mood_cooldown=48):
        self.fast_mode = fast_mode
        self.num_simulated_paths = num_simulated_paths if num_simulated_paths is not None \
            else (10_000 if fast_mode else 100_000)
        self.max_holding_period = max_holding_period
        self.ou_ols_window = ou_ols_window
        self.max_positions = max_positions
        self.weight_per_position = weight_per_position
        self.max_new_entries_per_bar = max_new_entries_per_bar
        self._pi_plus_grid  = pi_plus_grid  if pi_plus_grid  is not None else _PI_PLUS_GRID.copy()
        self._pi_minus_grid = pi_minus_grid if pi_minus_grid is not None else _PI_MINUS_GRID.copy()
        self.min_pi_width = min_pi_width
        self.sigma_percentile = sigma_percentile
        self.mood_threshold = mood_threshold
        self.mood_warmup = mood_warmup
        self.mood_cooldown = mood_cooldown
        self.open_trades: dict = {}
        self.trade_log: list = []
        self.decision_log: list = []
        self._event_this_bar = False
        self._n_entries = 0; self._n_exits = 0; self._n_ou_skipped = 0
        self._mood_buffers: dict = {}; self._last_changepoint: dict = {}

    def __repr__(self):
        return (f"OuTrendPullbackStrategy(fast={self.fast_mode},paths={self.num_simulated_paths},"
                f"hold={self.max_holding_period},ols={self.ou_ols_window},"
                f"maxpos={self.max_positions},w={self.weight_per_position})")

    def required_data(self):
        return {"prices": None, "volume": None, "entry_signal": None}

    def _log_decision(self, decision, t, asset, **fields):
        row = {"decision": decision, "t": t, "asset": asset}; row.update(fields)
        self.decision_log.append(row)

    def _check_mood_regime(self, asset, log_ret, t):
        if asset not in self._mood_buffers: self._mood_buffers[asset] = []
        self._mood_buffers[asset].append(log_ret)
        t = pd.Timestamp(t)
        if (asset in self._last_changepoint and
                (t - self._last_changepoint[asset]) < pd.Timedelta(hours=self.mood_cooldown)):
            return False
        if len(self._mood_buffers[asset]) <= self.mood_warmup: return True
        if len(self._mood_buffers[asset]) > 500:
            self._mood_buffers[asset] = self._mood_buffers[asset][-500:]
        buf_arr = np.array(self._mood_buffers[asset], dtype=np.float64)
        dmax, tau = _mood_d_max_tau(buf_arr, len(buf_arr))
        if dmax > self.mood_threshold:
            self._last_changepoint[asset] = t
            self._mood_buffers[asset] = self._mood_buffers[asset][int(tau):]
            return False
        return True

    def _entry_context(self, data, asset):
        ctx = {}
        for name in ("entry_signal", "trend_up", "pullback", "recent_down", "roll_trend", "log_returns"):
            try:
                f = data.feature(name).iloc[-1]
                if asset in f.index:
                    v = f[asset]
                    if np.isfinite(v): ctx[name] = float(v)
            except (KeyError, AttributeError, TypeError): pass
        return ctx

    @staticmethod
    def _fit_trend(prices):
        if not HAS_NUMBA:
            s = np.asarray(prices, dtype=float)
            if len(s) < 10 or np.isnan(s).any() or np.any(s <= 0): return None
            deltas = np.diff(s); mu = float(np.mean(deltas)); sigma = float(np.std(deltas, ddof=1))
            if not np.isfinite(mu) or not np.isfinite(sigma) or sigma <= 0: return None
            return 'abm', mu, sigma
        s = prices.astype(np.float64)
        mtype = _detect_trend_type(s)
        result = _fit_gbm_numba(s) if mtype == 'gbm' else _fit_abm_numba(s)
        if result is None or np.isnan(result[0]) or np.isnan(result[1]): return None
        return mtype, float(result[0]), float(result[1])

    def _calibrate_thresholds(self, model_type_str, mu, sigma, p0, seed):
        mt = 1 if model_type_str == "gbm" else 0
        args = (float(mu), float(sigma), float(p0), int(self.num_simulated_paths),
                int(self.max_holding_period), int(seed), self._pi_plus_grid, self._pi_minus_grid, mt)
        if HAS_GPU: return _calibrate_trend_gpu(*args)
        if not HAS_NUMBA: raise RuntimeError("Numba or GPU required")
        return _calibrate_trend_fused(*args)

    def _try_enter(self, asset, prices, t, ctx=None):
        ctx = dict(ctx or {})
        log_ret = ctx.get("log_returns", np.nan)
        if np.isfinite(log_ret):
            if not self._check_mood_regime(asset, log_ret, t):
                self._n_ou_skipped += 1
                self._log_decision("entry_reject", t, asset, reason="regime_changepoint", **ctx)
                return False
        hist = prices[asset].dropna()
        if len(hist) < self.ou_ols_window:
            self._n_ou_skipped += 1
            self._log_decision("entry_reject", t, asset, reason="short_history",
                               hist_len=int(len(hist)), need=self.ou_ols_window, **ctx)
            return False
        window = hist.iloc[-self.ou_ols_window:]; p0 = float(window.iloc[-1])
        trend_fit = self._fit_trend(window.values)
        if trend_fit is None:
            self._n_ou_skipped += 1
            self._log_decision("entry_reject", t, asset, reason="trend_fit_fail", p0=p0, **ctx)
            return False
        model_type, mu, sigma = trend_fit
        if mu <= 0:
            self._n_ou_skipped += 1
            self._log_decision("entry_reject", t, asset, reason="negative_drift",
                               mu=float(mu), model_type=model_type, **ctx)
            return False
        snr = mu/sigma if sigma > 0 else 0
        if snr < 0.05:
            self._n_ou_skipped += 1
            self._log_decision("entry_reject", t, asset, reason="low_signal_to_noise",
                               snr=float(snr), mu=float(mu), sigma=float(sigma),
                               model_type=model_type, **ctx)
            return False
        window_sigmas = []
        hist_available = len(hist)
        for i in range(max(self.ou_ols_window, hist_available - 500), hist_available, 20):
            if i < self.ou_ols_window: continue
            mini = hist.iloc[i - self.ou_ols_window: i]
            if len(mini) >= self.ou_ols_window:
                ft = self._fit_trend(mini.values)
                if ft is not None: window_sigmas.append(ft[2])
        if len(window_sigmas) >= 10:
            sig_thresh = float(np.percentile(window_sigmas, 75))
            if sigma > sig_thresh:
                self._n_ou_skipped += 1
                self._log_decision("entry_reject", t, asset, reason="sigma_too_high",
                                   sigma=float(sigma), threshold=sig_thresh,
                                   n_samples=len(window_sigmas), model_type=model_type, **ctx)
                return False
        seed = hash((asset, str(t))) % (2**32)
        cal = self._calibrate_thresholds(model_type, mu, sigma, p0, seed)
        pi_plus, pi_minus, ou_sharpe = cal
        if not (np.isfinite(pi_plus) and np.isfinite(pi_minus) and np.isfinite(ou_sharpe)):
            self._n_ou_skipped += 1
            self._log_decision("entry_reject", t, asset, reason="ou_calibrate_fail",
                               p0=p0, model_type=model_type, mu=float(mu), sigma=float(sigma), **ctx)
            return False
        if ou_sharpe < 0.0:
            self._n_ou_skipped += 1
            self._log_decision("entry_reject", t, asset, reason="ou_sharpe_too_low",
                               ou_sharpe=float(ou_sharpe), **ctx)
            return False
        pi_width = pi_plus - pi_minus
        if pi_width < self.min_pi_width:
            self._n_ou_skipped += 1
            self._log_decision("entry_reject", t, asset, reason="corridor_too_tight",
                               pi_width=float(pi_width), min_required=self.min_pi_width,
                               pi_plus=float(pi_plus), pi_minus=float(pi_minus), **ctx)
            return False
        trade = {"entry_t": t, "p0": p0, "model_type": model_type, "mu": float(mu),
                 "sigma": float(sigma), "pi_plus": float(pi_plus), "pi_minus": float(pi_minus),
                 "ou_sharpe": float(ou_sharpe), "bars_held": 0, "entry_ctx": ctx}
        self.open_trades[asset] = trade
        self._n_entries += 1; self._event_this_bar = True
        self._log_decision("entry_open", t, asset, reason="accepted", p0=p0,
                           model_type=model_type, mu=float(mu), sigma=float(sigma),
                           snr=float(snr), pi_plus=float(pi_plus), pi_minus=float(pi_minus),
                           pi_width=float(pi_width), ou_sharpe=float(ou_sharpe),
                           weight=self.weight_per_position, **ctx)
        return True

    def generate_weights(self, data, t):
        self._event_this_bar = False
        prices = data.prices
        entry_row = data.feature("entry_signal").iloc[-1]
        eligible = set(data.assets)
        weights = pd.Series(0.0, index=prices.columns)
        for asset in list(self.open_trades.keys()):
            if asset not in eligible:
                weights[asset] = self.weight_per_position; continue
            col = prices[asset] if asset in prices.columns else None
            if col is None or col.empty:
                weights[asset] = self.weight_per_position; continue
            p = float(col.iloc[-1])
            if not np.isfinite(p):
                weights[asset] = self.weight_per_position; continue
            trade = self.open_trades[asset]; trade["bars_held"] += 1
            pi_t = float(p) - trade["p0"]
            timeout_limit = 48 if trade.get("ou_sharpe", 0) < 0.0 else self.max_holding_period
            exit_now = pi_t >= trade["pi_plus"] or pi_t <= trade["pi_minus"] \
                       or trade["bars_held"] >= timeout_limit
            if exit_now:
                reason = ("pi_plus" if pi_t >= trade["pi_plus"]
                          else "pi_minus" if pi_t <= trade["pi_minus"] else "timeout")
                ret_pct = pi_t / trade["p0"] if trade["p0"] else np.nan
                self.trade_log.append({
                    "asset": asset, "entry_t": trade["entry_t"], "exit_t": t,
                    "bars_held": trade["bars_held"], "p0": trade["p0"], "p_exit": float(p),
                    "model_type": trade["model_type"], "mu": trade["mu"], "pi_exit": pi_t,
                    "ret_pct": float(ret_pct), "win": bool(pi_t > 0), "exit_reason": reason,
                    "sigma": trade["sigma"], "pi_plus": trade["pi_plus"],
                    "pi_minus": trade["pi_minus"], "ou_sharpe": trade["ou_sharpe"],
                    "entry_ctx": trade.get("entry_ctx"),
                })
                self._log_decision("exit", t, asset, reason=reason, p_exit=float(p),
                                   pi_exit=pi_t, ret_pct=float(ret_pct), win=bool(pi_t > 0),
                                   bars_held=trade["bars_held"], pi_plus=trade["pi_plus"],
                                   pi_minus=trade["pi_minus"],
                                   **(trade.get("entry_ctx") or {}))
                del self.open_trades[asset]; self._n_exits += 1; self._event_this_bar = True
            else:
                weights[asset] = self.weight_per_position
        new_entries = 0
        for asset in eligible:
            if asset in self.open_trades: continue
            sig_val = float(entry_row.get(asset, 0.0)) if asset in entry_row.index else 0.0
            if not (np.isfinite(sig_val) and sig_val > 0.5): continue
            ctx = self._entry_context(data, asset)
            if len(self.open_trades) >= self.max_positions:
                self._log_decision("entry_reject", t, asset, reason="max_positions",
                                   n_open=len(self.open_trades), cap=self.max_positions, **ctx)
                continue
            if new_entries >= self.max_new_entries_per_bar:
                self._log_decision("entry_reject", t, asset, reason="max_new_per_bar",
                                   cap=self.max_new_entries_per_bar, **ctx)
                continue
            if self._try_enter(asset, prices, t, ctx):
                weights[asset] = self.weight_per_position; new_entries += 1
        return weights

    def apply_risk(self, proposed, state, data):
        equity = state["equity"]
        if not self._event_this_bar and equity > 0:
            proposed = state["positions"] / equity
        return super().apply_risk(proposed, state, data)


# â”€â”€ CryptoRegimeOuTrendStrategy â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

class CryptoRegimeOuTrendStrategy(OuTrendPullbackStrategy):
    """
    OuTrendPullbackStrategy gated by a BTC DVOL expanding-percentile macro regime filter.
    DVOL > expanding P(dvol_percentile) of all DVOL seen so far â†’ regime OFF: close all
    positions immediately and block new entries until DVOL falls back below the threshold.
    """
    rebalance_frequency = "D"
    risk = RiskConfig(max_position=None, max_gross=1.0, max_net=1.0)

    def __init__(self, dvol_percentile=DVOL_PERCENTILE, dvol_warmup=DVOL_WARMUP_BARS, **kwargs):
        super().__init__(**kwargs)
        self.dvol_percentile = dvol_percentile
        self.dvol_warmup     = dvol_warmup
        self._dvol_hist: list = []   # grows bar-by-bar; PIT-safe expanding history

    def __repr__(self):
        return (f"CryptoRegimeOuTrendStrategy("
                f"dvol_pct={self.dvol_percentile}, dvol_warmup={self.dvol_warmup}, "
                f"fast={self.fast_mode}, paths={self.num_simulated_paths}, "
                f"hold={self.max_holding_period}, ols={self.ou_ols_window}, "
                f"maxpos={self.max_positions}, w={self.weight_per_position})")

    def required_data(self):
        return {**super().required_data(), "DVOL": None}

    def _close_all_for_regime_off(self, prices, t):
        for asset in list(self.open_trades.keys()):
            trade = self.open_trades[asset]
            if asset in prices.columns and not prices[asset].empty:
                p = float(prices[asset].iloc[-1])
            else:
                p = trade["p0"]
            if not np.isfinite(p):
                p = trade["p0"]
            pi_t    = p - trade["p0"]
            ret_pct = pi_t / trade["p0"] if trade["p0"] else np.nan
            self.trade_log.append({
                "asset": asset, "entry_t": trade["entry_t"], "exit_t": t,
                "bars_held": trade.get("bars_held", 0), "p0": trade["p0"], "p_exit": p,
                "model_type": trade.get("model_type", "unknown"), "mu": trade.get("mu", np.nan),
                "pi_exit": pi_t, "ret_pct": float(ret_pct), "win": bool(pi_t > 0),
                "exit_reason": "regime_off", "sigma": trade.get("sigma", np.nan),
                "pi_plus": trade.get("pi_plus", np.nan), "pi_minus": trade.get("pi_minus", np.nan),
                "ou_sharpe": trade.get("ou_sharpe", np.nan), "entry_ctx": trade.get("entry_ctx"),
            })
            self._log_decision("exit", t, asset, reason="regime_off", p_exit=p,
                               pi_exit=pi_t, ret_pct=float(ret_pct), win=bool(pi_t > 0),
                               bars_held=trade.get("bars_held", 0))
            del self.open_trades[asset]
            self._n_exits += 1

    def generate_weights(self, data, t):
        self._event_this_bar = False

        # â”€â”€ Macro regime gate (Layer 1) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
        dvol_row     = data.feature("DVOL").iloc[-1]
        current_dvol = float(dvol_row.iloc[0]) if len(dvol_row) > 0 else np.nan

        if np.isfinite(current_dvol):
            self._dvol_hist.append(current_dvol)

        if len(self._dvol_hist) >= self.dvol_warmup and np.isfinite(current_dvol):
            expanding_threshold = float(np.percentile(self._dvol_hist, self.dvol_percentile))
            regime_off = current_dvol > expanding_threshold
        else:
            regime_off = False   # still in warmup â€” allow trading

        if regime_off:
            # Block new entries; existing trades run to their natural OU exit
            _saved = self.max_new_entries_per_bar
            self.max_new_entries_per_bar = 0
            weights = super().generate_weights(data, t)
            self.max_new_entries_per_bar = _saved
            return weights

        # â”€â”€ Regime ON: delegate to OU trend+pullback layer (Layer 2) â”€â”€â”€â”€â”€â”€â”€â”€â”€
        return super().generate_weights(data, t)


# â”€â”€ Instantiate + validate â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
strat = CryptoRegimeOuTrendStrategy()
print(strat)
print(f"\nrequired_data : {strat.required_data()}")

rb = strat.rebalance_dates(panel.dates)
print(f"Rebalance bars: {len(rb):,}  (every hourly bar â€” 'D' = every bar in dates)")
print(f"Rebalances/day: {len(rb) / (len(panel.dates) / 24):.1f}")
print(f"\nNumba : {HAS_NUMBA}   GPU : {HAS_GPU}   Cores : {N_CORES}")
print(f"Paths : {strat.num_simulated_paths:,}   Grid : "
      f"{len(strat._pi_plus_grid)}Ã—{len(strat._pi_minus_grid)} = "
      f"{len(strat._pi_plus_grid)*len(strat._pi_minus_grid)} nodes")
print(f"\nDVOL gate : pct={strat.dvol_percentile}  warmup={strat.dvol_warmup} bars")

# Numba warmup (JIT compile)
print("\nWarming up Numba JIT (first call compiles; cache=True means once only) â€¦")
_t0 = time.perf_counter()
_ = _calibrate_trend_fused(0.5, 0.01, 100.0, 100, 10, 42, _PI_PLUS_GRID[:3], _PI_MINUS_GRID[:3], 0)
print(f"Done in {time.perf_counter()-_t0:.1f}s")


In [ ]:
# Cell 4 — Costs + Liquidity
from backtest.costs import (
    Commission, Spread, MarketImpact, ShortBorrow,
    CompositeCostModel, LiquidityCap,
)

costs = CompositeCostModel([
    Commission(bps=10),                                        # 10 bps per executed trade
    Spread(half_bps=5),                                        # 5 bps half-spread
    MarketImpact(k=10, kind="sqrt", adv_lookback=20),          # sqrt Almgren-style, 20-bar ADV
    ShortBorrow(annual_bps=300, trading_days=6048),             # 6048 = 252d × 24h; long-only → zero charge
])

liq = LiquidityCap(cap_pct=0.10, adv_lookback=20)              # cap each trade to 10% of 20-bar ADV

print("Cost components :", [type(m).__name__ for m in costs.models])
print(f"LiquidityCap    : cap_pct={liq.cap_pct}, adv_lookback={liq.adv_lookback}")
print(f"Panel has volume: {panel.has_field('volume')}")


In [ ]:
# Cell 5 — Single-path Engine run
from backtest.engine import Engine

WARMUP_BARS    = 720          # 30d × 24h: DVOL gate warmup + first PIT universe month
INITIAL_CAP    = 1_000_000

engine = Engine(
    panel,
    strat,
    costs=costs,
    liquidity=liq,
    initial_capital=INITIAL_CAP,
)
result = engine.run(start=panel.dates[WARMUP_BARS])

print(f"Bars          : {len(result.equity_curve):,}")
print(f"Rebalances    : {result.metadata['n_rebalances']:,}")
print(f"Date range    : {result.metadata['start'].date()} → {result.metadata['end'].date()}")
print(f"Initial       : ${result.metadata['initial_capital']:>14,.0f}")
print(f"Final equity  : ${result.equity_curve.iloc[-1]:>14,.0f}")
print(f"Total return  : {(result.equity_curve.iloc[-1] / INITIAL_CAP - 1):+.2%}")
print(f"Total costs   : ${result.costs.sum():>14,.0f}")
print(f"Max pos weight: {result.weights.abs().max().max():.2%}")
print(f"Max gross exp : {result.weights.abs().sum(axis=1).max():.2%}")


In [ ]:
# Cell 6 — CPCV splitter + MultiPathEngine
from backtest.splitters import CombinatorialPurgedCV
from backtest.engine import MultiPathEngine

# ── Splitter ──────────────────────────────────────────────────────────────────
splitter = CombinatorialPurgedCV(
    n_splits=6,
    n_test_groups=2,
    purge_bars=168,       # 7d × 24h — matches longest feature lookback
    embargo_pct=0.01,
)

n = len(panel.dates)
embargo_bars = int(n * splitter.embargo_pct)
print(f"Timeline            : {n:,} bars  ({panel.dates[0].date()} → {panel.dates[-1].date()})")
print(f"Groups              : {splitter.n_groups} of ~{n // splitter.n_groups:,} bars each (~{n // splitter.n_groups / 24:.0f} days)")
print(f"Splits (trainings)  : {splitter.n_splits()}")
print(f"Paths (curves)      : {splitter.n_paths()}")
print(f"Purge               : {splitter.purge_bars} bars (leading)")
print(f"Embargo             : {embargo_bars} bars (trailing-only; trailing total = {splitter.purge_bars + embargo_bars})")

first_train, first_test = next(iter(splitter.split(panel.dates)))
print(f"First split         : train={len(first_train):,}  test={len(first_test):,}  "
      f"disjoint={len(first_train.intersection(first_test)) == 0}")

# ── MultiPathEngine — factory so each fold starts with clean strategy state ───
strat_factory = lambda: CryptoRegimeOuTrendStrategy()

mp = MultiPathEngine(
    panel,
    strat_factory,
    splitter,
    costs=costs,
    liquidity=liq,
    initial_capital=1_000_000,
)

print(f"\nRunning {splitter.n_splits()} splits (n_jobs=-1) …")
res = mp.run(n_jobs=-1)

print(f"\nSplits run          : {res.n_splits}")
print(f"Paths assembled     : {res.n_paths}")
print(f"path_returns shape  : {res.path_returns.shape}")
print(f"path_equity shape   : {res.path_equity.shape}")
print(f"Date range          : {res.path_returns.index[0].date()} → {res.path_returns.index[-1].date()}")
print(f"Per-path NaN count  : min={res.path_returns.isna().sum().min()},  max={res.path_returns.isna().sum().max()}")
print()
print("Final equity per path:")
print(res.path_equity.iloc[-1].rename("final_equity").map(lambda v: f"${v:,.0f}"))


In [ ]:
# Cell 7 — Metrics
# ann_factor = 8760 (24h × 365d): crypto trades 24/7, hourly bars

ANN_FACTOR = 8760

# ── Single-path report + dashboard ───────────────────────────────────────────
print("=" * 60)
print("SINGLE-PATH (full sample)")
print("=" * 60)
single_rep = result.summary(ann_factor=ANN_FACTOR)

# ── Multi-path aggregate ──────────────────────────────────────────────────────
print("\n" + "=" * 60)
print(f"CPCV ({res.n_paths} paths)  —  aggregate metrics")
print("=" * 60)
agg      = res.aggregate_metrics()
per_path = res.metrics_per_path()

_rows = ["sharpe", "sortino", "calmar", "max_drawdown", "psr"]
_cols = ["mean", "std", "min", "max"]
_hdr  = f"{'metric':<20}" + "".join(f"{c:>10}" for c in _cols)
print(_hdr)
print("-" * len(_hdr))
for row in _rows:
    vals = [agg.get(f"{row}_{c}", float("nan")) for c in _cols]
    print(f"{row:<20}" + "".join(f"{v:>10.3f}" if not (v != v) else f"{'NaN':>10}" for v in vals))

print()
print(f"{'path':<10}{'sharpe':>10}{'ann_ret':>10}{'max_dd':>10}{'psr':>10}")
print("-" * 42)
for i, rep in enumerate(per_path):
    print(f"{'path_'+str(i):<10}{rep.sharpe:>10.3f}{rep.ann_return:>10.2%}"
          f"{rep.max_drawdown:>10.2%}{rep.psr:>10.3f}")

print()
print(f"Sharpe mean ± std : {agg['sharpe_mean']:.3f} ± {agg['sharpe_std']:.3f}")
print(f"PSR(0) mean       : {agg['psr_mean']:.3f}  "
      f"({'all paths significant' if agg['psr_min'] > 0.95 else 'not all paths significant'} at 95%)")
print(f"Max drawdown mean : {agg['max_drawdown_mean']:.2%}")


In [ ]:
# Cell 8 — DSR / Selection
# K = 1: only strategy tried this session.
# CPCV paths are one trial measured 5 ways, not 5 independent trials.

import numpy as np
from backtest.selection import dsr, effective_k, expected_max_sr
from backtest.metrics.core import sharpe_var_term

ANN_FACTOR = 8760   # hourly crypto (24h × 365d)
K          = 1      # single trial this session

# ── Per-bar SR variance from CPCV paths (empirical override) ─────────────────
path_sharpes_ann = np.array([rep.sharpe for rep in per_path])
path_sharpes_pb  = path_sharpes_ann / np.sqrt(ANN_FACTOR)
sr_var_empirical = float(np.var(path_sharpes_pb, ddof=1))   # per-bar variance

print(f"CPCV path Sharpes (ann.): {np.round(path_sharpes_ann, 3).tolist()}")
print(f"Empirical per-bar SR var : {sr_var_empirical:.6f}")

# ── DSR on each CPCV path ─────────────────────────────────────────────────────
print(f"\nDSR per CPCV path  (K={K}, empirical sr_variance)")
print(f"{'path':<10}{'sharpe_ann':>12}{'DSR':>10}")
print("-" * 34)
path_dsr_vals = []
for col in res.path_returns.columns:
    ret = res.path_returns[col].dropna()
    d   = dsr(ret, K=K, sr_variance=sr_var_empirical)
    path_dsr_vals.append(d)
    sr_ann = ret.mean() / ret.std(ddof=1) * np.sqrt(ANN_FACTOR)
    print(f"{col:<10}{sr_ann:>12.3f}{d:>10.3f}")

print(f"\nMedian DSR : {np.nanmedian(path_dsr_vals):.3f}")
print(f"Min DSR    : {np.nanmin(path_dsr_vals):.3f}")
print(f"Max DSR    : {np.nanmax(path_dsr_vals):.3f}")

# ── Single-path DSR (reference) ───────────────────────────────────────────────
r_single   = result.returns.dropna().to_numpy()
T          = len(r_single)
sr_pb      = r_single.mean() / r_single.std(ddof=1)
var_term   = sharpe_var_term(r_single, sr_pb)
luck_max_pb  = expected_max_sr(K, var_term / T)
luck_max_ann = luck_max_pb * np.sqrt(ANN_FACTOR)
dsr_single = dsr(result.returns, K=K, sr_variance=sr_var_empirical)

print(f"\n── Single-path reference (K={K}) ──")
print(f"Realized Sharpe (ann.) : {sr_pb * np.sqrt(ANN_FACTOR):.3f}")
print(f"Luck-max threshold     : {luck_max_ann:.4f}  (E[max SR | K={K}])")
print(f"DSR (empirical var)    : {dsr_single:.3f}")
print(f"PSR(0) [= DSR at K=1]  : {single_rep.psr:.3f}")
sig = "significant" if dsr_single > 0.95 else "NOT significant"
print(f"→ {sig} at 95% (DSR = {dsr_single:.3f})")

# ── Effective-K on CPCV paths (transparency) ─────────────────────────────────
k_eff = effective_k(res.path_returns.dropna(), threshold=0.5)
print(f"\neffective_k(path_returns, threshold=0.5) = {k_eff}  "
      f"(paths are correlated variants of one trial — K stays {K})")
